## CONSTRUCTION DE LA DIMENSION TEMPS

In [5]:
# importation des bibliotheques
import  pandas as pd
from pathlib import Path

In [6]:
# chemin vers les dataset
data_dir=Path("../data/raw")

#chargement dees commandes pour la recuperation des dates
commandes=pd.read_csv(data_dir/"olist_orders_dataset.csv")

In [7]:
# appercu du dataset
commandes.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [8]:
# conversion des dates en datetimes
commandes["order_purchase_timestamp"]=pd.to_datetime(commandes["order_purchase_timestamp"])

print("total commande:", len(commandes))
print("date minimum:", commandes["order_purchase_timestamp"].min())
print("date maximale :", commandes["order_purchase_timestamp"].max())

total commande: 99441
date minimum: 2016-09-04 21:15:19
date maximale : 2018-10-17 17:30:18


###
gereration de DimTemps

In [9]:
# recuperation de la plage des dates
date_min=commandes["order_purchase_timestamp"].min().date()
date_max=commandes["order_purchase_timestamp"].max().date()

print("plage:",date_min ,"-",date_max)

plage: 2016-09-04 - 2018-10-17


In [12]:
# generation d'une ligne par jour
date=pd.date_range(start=date_min,end=date_max,freq="D")

# construction du dataframe
dim_temps=pd.DataFrame({
    "date":date.date,
    "annee": date.year,
    "trimestre": date.quarter,
    "mois": date.month,
    "jour_semaine":date.dayofweek+1
})

In [15]:
# appercu du dataframe
print("nombres total d'observation :", len(dim_temps))
print("information sur les colonnes :",list(dim_temps.columns))
print("les 5 premieres lignes:\n")
print(dim_temps.head())

nombres total d'observation : 774
information sur les colonnes : ['date', 'annee', 'trimestre', 'mois', 'jour_semaine']
les 5 premieres lignes:

         date  annee  trimestre  mois  jour_semaine
0  2016-09-04   2016          3     9             7
1  2016-09-05   2016          3     9             1
2  2016-09-06   2016          3     9             2
3  2016-09-07   2016          3     9             3
4  2016-09-08   2016          3     9             4


###
verification des assertions

In [14]:
# unicite de la colonne
assert dim_temps["date"].is_unique, "dates dupliquees"
# pas de NA
assert  dim_temps.isna().sum().sum()==0, "valeurs manquantes"
# coherence du jour de la semaine
assert dim_temps["jour_semaine"].between(1,7).all(),"jour invalide"
# coherence de la valeur du mois
assert dim_temps["mois"].between(1,12).all()," mois invalide"
# coherence du trimestre:
assert dim_temps["trimestre"].between(1,4).all(),"trimestre invalide"
# coherence du nombre de ligne
assert len(dim_temps)>700,f"peu de ligne,{len(dim_temps)}"

print(" controle de la dimension Temps:\n")
print("total ligne:", len(dim_temps))
print("plage:", dim_temps["date"].min(),"-",dim_temps["date"].max())
print("annees :", sorted(dim_temps["annee"].unique()))

 controle de la dimension Temps:

total ligne: 774
plage: 2016-09-04 - 2018-10-17
annees : [np.int32(2016), np.int32(2017), np.int32(2018)]


## CONSTRUCTION DE LA DIMENSION  TEMPS TERMINEE